# Prompt Chaining

**Module:** 07-prompt-engineering

**Notebook:** `06-prompt-chaining.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **What is Prompt Chaining?** with clear contracts and failure modes
- Explain and apply **Sequential Chains** with clear contracts and failure modes
- Explain and apply **Multi-Step Reasoning Chains** with clear contracts and failure modes
- Explain and apply **Pipeline Pattern** with clear contracts and failure modes
- Explain and apply **Intermediate Results** with clear contracts and failure modes
- Explain and apply **Error Handling** with clear contracts and failure modes
- Explain and apply **When Not to Chain** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — Prompt Chaining

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **What is Prompt Chaining?**
2. **Sequential Chains**
3. **Multi-Step Reasoning Chains**
4. **Pipeline Pattern**
5. **Intermediate Results**
6. **Error Handling**
7. **When Not to Chain**

Read top-to-bottom once, then revisit weak spots with the exercises.


## What is Prompt Chaining?

### Definition
**What is Prompt Chaining?** is a core building block in 06-prompt-chaining within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around What is Prompt Chaining? typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For What is Prompt Chaining?: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain What is Prompt Chaining? as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating What is Prompt Chaining? as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for What is Prompt Chaining?
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use What is Prompt Chaining? when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does What is Prompt Chaining? improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "What is Prompt Chaining?" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "What is Prompt Chaining?"
    notebook: str = "06-prompt-chaining"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


## Sequential Chains

### Definition
**Sequential Chains** is a core building block in 06-prompt-chaining within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Sequential Chains typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Sequential Chains: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Sequential Chains as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Sequential Chains as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Sequential Chains
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Sequential Chains when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Sequential Chains" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Sequential Chains"
    notebook: str = "06-prompt-chaining"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


### Worked scenario — Sequential Chains

**Situation:** A team wants to productionize a feature involving **Sequential Chains**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Multi-Step Reasoning Chains

### Definition
**Multi-Step Reasoning Chains** is a core building block in 06-prompt-chaining within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Multi-Step Reasoning Chains typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Multi-Step Reasoning Chains: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Multi-Step Reasoning Chains as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Multi-Step Reasoning Chains as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Multi-Step Reasoning Chains
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Multi-Step Reasoning Chains when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Multi-Step Reasoning Chains" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Multi-Step Reasoning Chains"
    notebook: str = "06-prompt-chaining"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


## Pipeline Pattern

### Definition
**Pipeline Pattern** is a core building block in 06-prompt-chaining within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Pipeline Pattern typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Pipeline Pattern: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Pipeline Pattern as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Pipeline Pattern as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Pipeline Pattern
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Pipeline Pattern when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Pipeline Pattern" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Pipeline Pattern"
    notebook: str = "06-prompt-chaining"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


In [ ]:
# Demo: decision table for applying "Pipeline Pattern"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_pipeline_pat", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


### Worked scenario — Pipeline Pattern

**Situation:** A team wants to productionize a feature involving **Pipeline Pattern**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Intermediate Results

### Definition
**Intermediate Results** is a core building block in 06-prompt-chaining within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Intermediate Results typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Intermediate Results: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Intermediate Results as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Intermediate Results as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Intermediate Results
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Intermediate Results when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Intermediate Results" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Intermediate Results"
    notebook: str = "06-prompt-chaining"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Intermediate Results"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Intermediate Results"}
strong = {"definition": "Intermediate Results", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Intermediate Results"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Intermediate Results", "passed": len(checks)-len(failed), "failed": failed})


## Error Handling

### Definition
**Error Handling** is a core building block in 06-prompt-chaining within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around Error Handling typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Error Handling: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain Error Handling as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Error Handling as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Error Handling
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use Error Handling when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Error Handling" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Error Handling"
    notebook: str = "06-prompt-chaining"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Error Handling"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Error Handling"}
strong = {"definition": "Error Handling", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Error Handling"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Error Handling", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Error Handling

**Situation:** A team wants to productionize a feature involving **Error Handling**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## When Not to Chain

### Definition
**When Not to Chain** is a core building block in 06-prompt-chaining within prompt engineering. Treat it as a versioned API contract written in natural language: something you can name, version, test, and operate.

### Why it matters
In prompt engineering, weak designs around When Not to Chain typically surface as ambiguous asks, format drift, and injection via untrusted input. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For When Not to Chain: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like prompt templates, schemas, and eval sets.

### Intuition
Explain When Not to Chain as a versioned API contract written in natural language. If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating When Not to Chain as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for When Not to Chain
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of prompt engineering: ambiguous asks, format drift, and injection via untrusted input

### When to use
Use When Not to Chain when your product path depends on this concern in prompt engineering. Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "When Not to Chain" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "When Not to Chain"
    notebook: str = "06-prompt-chaining"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
class Pipeline:
    def __init__(self):
        self.stages = []
    def add(self, name, fn):
        self.stages.append((name, fn)); return self
    def run(self, data):
        audit = []
        for name, fn in self.stages:
            data = fn(data)
            audit.append({"stage": name, "type": type(data).__name__})
        return data, audit

result, audit = (
    Pipeline()
    .add("normalize", lambda s: s.strip().lower())
    .add("tokens", lambda s: s.split())
    .add("features", lambda toks: {"n": len(toks), "head": toks[:3]})
    .run("  SSO Login Loop  ")
)
print(result); print(audit)


In [ ]:
# Parallel fan-out / fan-in sketch
from concurrent.futures import ThreadPoolExecutor

def analyze(kind, text):
    return {"kind": kind, "len": len(text)}

text = "investigate checkout latency"
with ThreadPoolExecutor(max_workers=3) as ex:
    parts = list(ex.map(lambda k: analyze(k, text), ["security", "perf", "ux"]))
merged = {p["kind"]: p["len"] for p in parts}
print(merged)


## Comparison Snapshot

Use this table when reviewing designs in **Prompt Chaining**.

| Topic | Do | Don't |
|-------|----|-------|
| What is Prompt Chaining? | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Sequential Chains | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Multi-Step Reasoning Chains | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Pipeline Pattern | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Intermediate Results | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Error Handling | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| What is Prompt Chaining? | Key concept covered in this notebook; see its section for definition and pitfalls |
| Sequential Chains | Key concept covered in this notebook; see its section for definition and pitfalls |
| Multi-Step Reasoning Chains | Key concept covered in this notebook; see its section for definition and pitfalls |
| Pipeline Pattern | Key concept covered in this notebook; see its section for definition and pitfalls |
| Intermediate Results | Key concept covered in this notebook; see its section for definition and pitfalls |
| Error Handling | Key concept covered in this notebook; see its section for definition and pitfalls |
| When Not to Chain | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **Prompt Chaining** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **07-prompt-engineering**.


## Try It Yourself

1. Implement a failing test/fixture for **What is Prompt Chaining?**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Sequential Chains**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Multi-Step Reasoning Chains**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Pipeline Pattern**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Intermediate Results**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
